# Phase 1 — PDF Ingestion

**Goal:** Load a research paper, extract text page-by-page with metadata, and emit a structured `Document` schema that downstream chunking and retrieval can use.

We use **PyMuPDF** (the `fitz` module) as the primary parser because it is the fastest pure-Python option that exposes both text and images with page-accurate coordinates — exactly what we need to support citations.

By the end you will have:

1. A canonical sample paper downloaded into `data/raw/`.
2. A `PdfDocument` Pydantic model with one entry per page.
3. A reusable `load_pdf(path) -> PdfDocument` function we can drop into `app/ingestion/pdf_loader.py`.
4. A first look at extracting embedded images (we will use these for figures in Phase 7).


## 1.1 — Download a sample paper

We will use *Attention Is All You Need* (Vaswani et al., 2017) — the canonical Transformer paper. It is a great test bed because it has:

- text, math, and tables;
- multiple figures including the famous architecture diagram (Fig. 1);
- realistic page count (~15 pages).


In [ ]:
import os, sys, urllib.request
from pathlib import Path

# Ensure we run from the repo root.
repo_root = Path.cwd()
while not (repo_root / "app").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
pdf_path = raw_dir / "attention_is_all_you_need.pdf"

if not pdf_path.exists():
    url = "https://arxiv.org/pdf/1706.03762"
    print("Downloading:", url)
    urllib.request.urlretrieve(url, pdf_path)

size_kb = pdf_path.stat().st_size / 1024
print(f"Saved {pdf_path} ({size_kb:.0f} KB)")


## 1.2 — Open the PDF and look at the structure


In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(pdf_path)
print("Pages           :", doc.page_count)
print("Metadata title  :", doc.metadata.get("title"))
print("Metadata author :", doc.metadata.get("author"))
print()
print("--- First page (first 400 chars) ---")
print(doc[0].get_text()[:400])


## 1.3 — Define the document schema

Citations only work if every piece of text *remembers* which document and page it came from. We model that explicitly with Pydantic:


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PageRecord(BaseModel):
    """One page of a PDF."""
    doc_id: str
    page: int                  # 1-indexed for human-readable citations
    text: str
    n_images: int = 0
    source: str                # original filename
    section: Optional[str] = None   # filled in by chunker later

class PdfDocument(BaseModel):
    doc_id: str
    source: str
    title: Optional[str] = None
    n_pages: int
    pages: list[PageRecord] = Field(default_factory=list)


## 1.4 — The `load_pdf` function

This is the function that lives in `app/ingestion/pdf_loader.py`. Keep it pure (no I/O surprises), return the structured object, and let callers decide what to do with it.


In [ ]:
import hashlib

def _doc_id(path: Path) -> str:
    h = hashlib.sha1(path.read_bytes()).hexdigest()[:10]
    return f"{path.stem}_{h}"

def load_pdf(path: str | Path) -> PdfDocument:
    path = Path(path)
    doc = fitz.open(path)
    doc_id = _doc_id(path)
    pages = []
    for i, page in enumerate(doc, start=1):
        text = page.get_text("text")  # plain text; "blocks" gives layout boxes
        n_images = len(page.get_images(full=True))
        pages.append(PageRecord(
            doc_id=doc_id,
            page=i,
            text=text,
            n_images=n_images,
            source=path.name,
        ))
    return PdfDocument(
        doc_id=doc_id,
        source=path.name,
        title=doc.metadata.get("title") or None,
        n_pages=doc.page_count,
        pages=pages,
    )

pdf = load_pdf(pdf_path)
print(f"Loaded {pdf.source} as {pdf.doc_id}")
print(f"  {pdf.n_pages} pages, total text chars = {sum(len(p.text) for p in pdf.pages):,}")
print(f"  pages with embedded images = {sum(1 for p in pdf.pages if p.n_images > 0)}")


## 1.5 — Inspect a single page


In [ ]:
import pandas as pd
df = pd.DataFrame([
    {"page": p.page, "chars": len(p.text), "images": p.n_images, "preview": p.text[:60].replace(chr(10), " ")}
    for p in pdf.pages
])
df.head(10)


## 1.6 — Extracting embedded images

We will do real figure work in Phase 7, but it is worth a peek now. Each `page.get_images(full=True)` tuple references an image stream in the PDF; we dereference it through the doc's xref table.


In [ ]:
from PIL import Image
import io

processed_dir = Path("data/processed/figures")
processed_dir.mkdir(parents=True, exist_ok=True)

saved = []
for page_index, page in enumerate(doc, start=1):
    for img_index, img in enumerate(page.get_images(full=True), start=1):
        xref = img[0]
        pix = fitz.Pixmap(doc, xref)
        if pix.n - pix.alpha > 3:   # CMYK -> RGB
            pix = fitz.Pixmap(fitz.csRGB, pix)
        img_bytes = pix.tobytes("png")
        out_path = processed_dir / f"{pdf.doc_id}_p{page_index}_img{img_index}.png"
        out_path.write_bytes(img_bytes)
        saved.append(out_path)
        pix = None

print(f"Saved {len(saved)} images to {processed_dir}")
for p in saved[:5]:
    print(" -", p.name)


### Display one image inline


In [ ]:
from IPython.display import Image as IPyImage, display
if saved:
    display(IPyImage(filename=str(saved[0])))
else:
    print("No embedded raster images in this PDF (some papers only have vector figures).")


## 1.7 — Edge cases worth handling in production

A robust `load_pdf` needs to deal with:

| Edge case                  | Detection / mitigation                                                              |
|----------------------------|--------------------------------------------------------------------------------------|
| Scanned (image-only) PDFs  | `page.get_text()` returns empty — fall back to `pytesseract` OCR.                    |
| Two-column layouts         | `page.get_text("blocks")` returns layout boxes; sort by `(x, y)` instead of `(y, x)`.|
| Headers / footers / page numbers | Filter blocks whose bbox is in the top/bottom 5% of the page.                  |
| Inline equations           | `get_text("text")` flattens math to garbage. Use `latex-ocr` or a math-aware parser. |
| Tables                     | `pdfplumber.extract_tables()` is more reliable than PyMuPDF for grid tables.         |

We will not implement all of these in the tutorial — but the design of `PageRecord` (with `section` and `n_images` as first-class fields) gives us room to extend.


## 1.8 — OCR fallback (only if text extraction is empty)

This cell is illustrative — it will be a no-op on *Attention Is All You Need* (which has embedded text). Try it on a scanned PDF to see it kick in.


In [ ]:
def ocr_page_if_needed(page: fitz.Page, dpi: int = 200) -> str:
    text = page.get_text().strip()
    if text:
        return text
    try:
        import pytesseract
        from PIL import Image
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        return pytesseract.image_to_string(img)
    except Exception as e:
        print("OCR fallback failed:", e)
        return ""

ocr_check = ocr_page_if_needed(doc[0])
print(f"Page 1 has {len(ocr_check):,} chars (no OCR needed).")


## What you learned

- How to download and open a real research paper with PyMuPDF.
- Why every page needs a stable `doc_id` + `page` for citations.
- The `PdfDocument` / `PageRecord` Pydantic schema we will use throughout the project.
- How to extract embedded raster images (foundation for Phase 7 figure extraction).
- Edge cases in PDF parsing and which library handles each.

## Exercises

1. Modify `load_pdf` to also store `page.get_text("blocks")` so chunking can use layout-aware splits.
2. Add a parameter `dehyphenate: bool = True` that joins words split across line breaks (e.g. `train-\ning` → `training`).
3. Test the function on a scanned PDF (e.g. an older paper from Google Scholar) and confirm the OCR fallback kicks in.

## Next: [Phase 2 — Chunking strategies](./2026-05-25-phase02-chunking-strategies.ipynb)
